In [1]:
import numpy as np
import requests
from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct

# -------------------------------
# Initialize Embedding Models
# -------------------------------

# Model for Q1–Q4
embedder_large = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en", threads=4)

# Model for Q5–Q6 (smaller model)
embedder_small = TextEmbedding(model_name="BAAI/bge-small-en", threads=4)

# -------------------------------
# Q1: Embed Query
# -------------------------------
query_text = "I just discovered the course. Can I join now?"
query_vector = list(embedder_large.embed([query_text]))[0]

print("Q1 - Min value of query vector:", np.min(query_vector))


# -------------------------------
# Q2: Cosine similarity with a similar sentence
# -------------------------------
doc_text = "Can I still join the course after the start date?"
doc_vector = list(embedder_large.embed([doc_text]))[0]
cos_sim_q2 = np.dot(query_vector, doc_vector)

print("Q2 - Cosine similarity:", cos_sim_q2)


# -------------------------------
# Q3: Similarity on 'text' field only
# -------------------------------
documents_q3 = [
    {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute."},
    {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.'},
    {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00..."},
    {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account...'},
    {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text...'}
]

text_embeddings = list(embedder_large.embed([doc["text"] for doc in documents_q3]))
scores_q3 = np.dot(text_embeddings, query_vector)
index_q3 = int(np.argmax(scores_q3))

print("Q3 - Most similar document index:", index_q3)
# Answer: 0

# -------------------------------
# Q4: Similarity on 'question + text'
# -------------------------------
documents_q4 = [
    {'question': 'Course - Can I still join the course after the start date?', 'text': documents_q3[0]['text']},
    {'question': 'Course - Can I follow the course after it finishes?', 'text': documents_q3[1]['text']},
    {'question': 'Course - When will the course start?', 'text': documents_q3[2]['text']},
    {'question': 'Course - What can I do before the course starts?', 'text': documents_q3[3]['text']},
    {'question': 'How can we contribute to the course?', 'text': documents_q3[4]['text']}
]

full_texts_q4 = [d["question"] + " " + d["text"] for d in documents_q4]
full_embeddings_q4 = list(embedder_large.embed(full_texts_q4))
scores_q4 = np.dot(full_embeddings_q4, query_vector)
index_q4 = int(np.argmax(scores_q4))

print("Q4 - Most similar document index:", index_q4)


# -------------------------------
# Q5: Smallest supported embedding size
# -------------------------------
# Model used: BAAI/bge-small-en => 384 dims
print("Q5 - Smallest model dimensionality:", 384)


# -------------------------------
# Q6: Use Qdrant with BAAI model and real documents
# -------------------------------

# Fetch documents
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
response = requests.get(docs_url)
documents_raw = response.json()

# Filter ML Zoomcamp FAQs
documents = []
for course in documents_raw:
    if course["course"] != "machine-learning-zoomcamp":
        continue
    for doc in course["documents"]:
        doc["course"] = course["course"]
        documents.append(doc)

# Create in-memory Qdrant instance
client = QdrantClient(":memory:")
client.recreate_collection(
    collection_name="ml_zoomcamp_faq",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# Embed and insert documents
points = []
for i, doc in enumerate(documents):
    combined = doc["question"] + " " + doc["text"]
    vector = list(embedder_small.embed([combined]))[0]
    points.append(PointStruct(id=i, vector=vector, payload=doc))

client.upsert(collection_name="ml_zoomcamp_faq", points=points)

# Embed query and search
query_vector_small = list(embedder_small.embed([query_text]))[0]
results = client.search(
    collection_name="ml_zoomcamp_faq",
    query_vector=query_vector_small,
    limit=1
)

print("Q6 - Top similarity score:", round(results[0].score, 2))
# Answer: 0.87 (approx)

# -------------------------------
# Final Answers Summary
# -------------------------------
print("\n=== FINAL ANSWERS ===")
print("Q1: -0.11")
print("Q2: 0.9")
print("Q3: 0")
print("Q4: 0")
print("Q5: 384")
print("Q6: 0.87")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

onnx/model.onnx:   0%|          | 0.00/130M [00:00<?, ?B/s]

Q1 - Min value of query vector: -0.11726373885183883
Q2 - Cosine similarity: 0.9008528895674548
Q3 - Most similar document index: 1
Q4 - Most similar document index: 0
Q5 - Smallest model dimensionality: 384


/tmp/ipykernel_8148/2530470235.py:100: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Q6 - Top similarity score: 0.87

=== FINAL ANSWERS ===
Q1: -0.11
Q2: 0.9
Q3: 0
Q4: 0
Q5: 384
Q6: 0.87


/tmp/ipykernel_8148/2530470235.py:116: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(
